# 批量分析流程：SLEAP 推理（top / side1 / side2）→ 2D 追踪 → 3D 重建

给一个实验列表（完整文件夹名，或 `P12`、`E12` 这样的组名），自动完成三个视角的推理，以及两套 3D：本仓库的 `reconstruct_from_slp`，和 medaka-3d 完整流程（m3dfull）。
本机和 saion 集群可以分工同时跑；推理一落地，两个 3D 监视器会自动接上。

| 步骤 | 做什么 | 在哪里跑 | 用时 |
|---|---|---|---|
| 1–3 | 设参数、定位路径、列出实验和时间估算 | notebook | 秒级 |
| 4 | **小测试**：1 个实验正中间 20 秒，三视角推理 + 3D | 本机 GPU，阻塞 | ~10 分钟 |
| 5 | 本机推理队列（后台）：逐个实验、逐个视角 | 本机 GPU | 每实验 ~9 h |
| 6 | 生成 saion 提交命令（复制到集群执行） | saion GPU | 并行 |
| 7 | 两个 3D 监视器（后台）：推理一完成就做旧 3D，并做新 3D（m3dfull） | 本机 CPU | 旧 3D ~10–30 min；新 3D ~2 h |
| 8 | 进度总表 | notebook | 随时重跑 |
| 9 | 停止后台任务（需要重启监视器 / 队列时用） | notebook | 秒级 |
| 10 | Rerun 交互式查看 3D 结果（三视角视频 + 2D 点 + 3D 轨迹） | 本机 | 1–2 分钟加载 |

**背后的脚本**（逻辑都在 `scripts/` 里，notebook 只负责调用）：
- `scripts/common/experiment_sets.py` — 组名 → 文件夹（`P12` = P 组第 1、2 轮所有天；按组/轮匹配，不按日期）
- `scripts/sleap/batch_bottomup_track.py` — 本机推理队列；已完成的视角自动跳过，可随时重启
- `scripts/reconstruction_3d/batch_reconstruct_3d.py` — 合并集群分块 → `track_simple_nn` → `reconstruct_from_slp`（round8 生产参数）
- `scripts/reconstruction_3d/medaka3d_batch.py` — 新 3D：c27 → c34 → 导出 `<标签>_m3dfull` 和审阅包；`--watch` 等三视角 `.slp` 齐了再开始
- `scripts/hpc/submit_experiment.sh` — 已支持 bottom-up（`TOP_BOTTOMUP` / `SIDE_BOTTOMUP`）

**换地址 / 换电脑**：仓库根目录自动查找；数据路径只在 `configs/paths.<用户名>.toml`（没有则 `paths.local.toml`）里改。
新地址要先 `uv sync --python 3.11`，并在 Jupyter 里选该仓库 `.venv` 的内核。

**注意**：
- 全长运行期间关掉 Windows 自动睡眠；重启 / 睡眠会中断后台任务（重新运行步骤 5、7 即可接着跑，未完成的视角 / 实验从头重跑；残留的锁和半截文件会自动识别，不用手动删）
- 换模型时 `ROUND_TAG` 也要一起换，否则旧结果会被当作“已完成”跳过
- 集群用的是 saion 上 `git pull` 下来的代码：本地改了 `scripts/hpc/` 要先 commit + push

## 步骤 1 — 参数

只需要改这一格。
- `LOCAL_SET`：本机跑的实验；`HPC_SET`：交给 saion 的实验（`"ALL"` = 全部，自动减去本机那部分和 `_P2` 第二段录像）
- **`LOCAL_SET` 一旦开跑就不要再改**：`HPC_SET` 是“全部减本机”，改了会让同一个实验两边都跑
- 本机速度约 1 个实验/9 小时，12 天最多 ~30 个，**其余必须交给集群**
- `RUN_LOCAL_QUEUE` / `RUN_3D_WATCHER` 为 `True` 时步骤 5 / 7 才真正启动后台任务（防止“全部运行”误启动）。步骤 7 会同时启动旧 3D 和新 3D（m3dfull）
- 换模型 = 同时改 `ROUND_TAG`、`TOP_MODEL`、`SIDE_MODEL`

In [1]:
LOCAL_SET = ["E12"]          # 本机：E 组 6 个实验（不含 E_2_1 的第二段 P2）
HPC_SET   = []             # 集群：这次不用（要用时改成 ["ALL"] = 其余全部）
TEST_EXPERIMENT = "20260112_P_1_1_FCMC"
TEST_SECONDS    = 20         # 小测试：取视频正中间这么多秒

ROUND_TAG     = "s14_bottomup_3node_reviewed"
TOP_MODEL     = "retrain_20260810_3node_reviewed/top/s14.bottomup_3node.reviewed_top_f24_ms64_paf8"    # 相对 sleap_models_dir
SIDE_MODEL    = "retrain_20260810_3node_reviewed/side/s14.bottomup_3node.reviewed_side_f24_ms64_paf8"
BATCH_SIZE    = 4      # 本机 16 GB：4 最稳（≥4 速度一样，12 会显存溢出）
EXPECTED_FPS  = 33.0   # 本机实测
HPC_BATCH_SIZE = 8
HPC_NUM_CHUNKS = 4
HPC_PARALLEL_GPUS = 8  # largegpu 分区本账号同时可用的 GPU 数（用于估算）

# saion 上看到的路径（Z:\ = /bucket/StephensU）
HPC_REPO        = "~/medaka-mating-pipeline"
HPC_MODELS_ROOT = "/bucket/StephensU/medaka/sleap_models"

PATHS_CONFIG    = None   # None = 自动选择
RUN_LOCAL_QUEUE = True
RUN_3D_WATCHER  = True

## 步骤 2 — 定位仓库根目录和本机路径配置

- 输入：当前目录、`configs/paths.<用户名>.toml` 或 `configs/paths.local.toml`
- 输出：`REPO`、`CFG`、`PATHS`；检查内核是否是本仓库 `.venv`

In [2]:
import os
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for d in [start, *start.parents]:
        if (d / "pyproject.toml").exists() and (d / "scripts" / "common" / "local_paths.py").exists():
            return d
    raise FileNotFoundError(f"找不到仓库根目录（从 {start} 向上没有 pyproject.toml）")


REPO = find_repo_root(Path.cwd().resolve())
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from scripts.common.local_paths import load_local_paths
from scripts.sleap.batch_bottomup_track import default_paths_config

CFG = Path(PATHS_CONFIG) if PATHS_CONFIG else default_paths_config()
PATHS = load_local_paths(CFG)
if PATHS is None:
    raise FileNotFoundError(f"路径配置不存在：{CFG}（从 configs/paths.example.toml 复制一份再改）")
for key in ("medaka_data_dir", "sleap_models_dir", "sleap_output_dir", "reconstruction_output_dir", "runs_dir"):
    value = getattr(PATHS, key)
    print(f"{key:26s} {'✅' if value and value.exists() else '❌ 不存在'} {value}")

print("仓库根目录 :", REPO)
print("路径配置   :", CFG)
print("Python     :", sys.executable)
if not Path(sys.executable).resolve().is_relative_to((REPO / ".venv").resolve()):
    print("⚠️ 当前内核不是本仓库的 .venv —— 请在 Jupyter 里切换内核")

LOG_DIR = PATHS.runs_dir / "batch_pipeline"

medaka_data_dir            ✅ \\bucket.oist.jp\bucket\StephensU\medaka\medaka_202512
sleap_models_dir           ✅ \\bucket.oist.jp\bucket\StephensU\medaka\sleap_models
sleap_output_dir           ✅ \\bucket.oist.jp\bucket\StephensU\medaka\sleap_output
reconstruction_output_dir  ✅ \\bucket.oist.jp\bucket\StephensU\medaka\reconstruction_3d
runs_dir                   ✅ \\bucket.oist.jp\bucket\StephensU\medaka\runs
仓库根目录 : C:\Users\ReiterU\Documents\GitHub\medaka-mating-pipeline
路径配置   : C:\Users\ReiterU\Documents\GitHub\medaka-mating-pipeline\configs\paths.ReiterU.toml
Python     : c:\Users\ReiterU\Documents\GitHub\medaka-mating-pipeline\.venv\Scripts\python.exe


## 步骤 3 — 实验列表与时间估算

- 输入：`LOCAL_SET`、`HPC_SET`
- 输出：`LOCAL_EXPS`、`HPC_EXPS`（两边不重复），以及估算时间
- 缺 `experiment_metadata.json`（或其中 `total_frames` 为空）的实验只打印警告、不计入估算（推理本身不依赖它）
- 集群估算按本机速度、不含排队，只是量级参考

In [3]:
import json

from scripts.common.experiment_sets import resolve_experiments

VIEWS = ("top_cam", "side1_cam", "side2_cam")
LOCAL_EXPS = resolve_experiments(LOCAL_SET, PATHS.medaka_data_dir)
HPC_EXPS = [e for e in resolve_experiments(HPC_SET, PATHS.medaka_data_dir) if e not in LOCAL_EXPS]


def read_views(exp):
    # experiment_metadata.json 里的 views 段；缺文件、缺字段或 total_frames 为空返回 None
    meta_path = PATHS.medaka_data_dir / exp / "experiment_metadata.json"
    try:
        views = json.loads(meta_path.read_text(encoding="utf-8"))["views"]
        return views if views and all(isinstance(v.get("total_frames"), int) for v in views.values()) else None
    except (OSError, KeyError, ValueError, AttributeError):
        return None


missing_meta = [e for e in LOCAL_EXPS + HPC_EXPS if read_views(e) is None]


def gpu_hours(exps):
    return sum(sum(v["total_frames"] for v in read_views(e).values())
               for e in exps if e not in missing_meta) / EXPECTED_FPS / 3600


local_h, hpc_h = gpu_hours(LOCAL_EXPS), gpu_hours(HPC_EXPS)
print(f"本机 {len(LOCAL_EXPS)} 个：{', '.join(LOCAL_EXPS)}")
print(f"  → 约 {local_h:.0f} GPU 小时 = {local_h / 24:.1f} 天")
print(f"集群 {len(HPC_EXPS)} 个")
print(f"  → 约 {hpc_h:.0f} GPU 小时；{HPC_PARALLEL_GPUS} 块 GPU 并行约 {hpc_h / HPC_PARALLEL_GPUS / 24:.1f} 天（不含排队，按本机速度估）")
if missing_meta:
    print(f"\n⚠️ {len(missing_meta)} 个实验缺 experiment_metadata.json（或其中没有 total_frames），未计入估算：")
    print("  " + "\n  ".join(missing_meta))

本机 6 个：20260112_P_1_1_FCMC, 20260112_P_1_3_FCMC, 20260113_P_1_2_FCMC, 20260118_P_2_1_FCMC, 20260118_P_2_2_FCMC, 20260118_P_2_3_FCMC
  → 约 53 GPU 小时 = 2.2 天
集群 0 个
  → 约 0 GPU 小时；8 块 GPU 并行约 0.0 天（不含排队，按本机速度估）


## 步骤 4 — 小测试（视频正中间 `TEST_SECONDS` 秒 × 3 视角 → 3D）

全长开跑前先确认整条流程能走通：3-node bottom-up 模型还没走过 3D 流程。
- 帧范围从 `experiment_metadata.json` 自动算：三个视角里最短的总帧数取中点，前后各取一半
- 输出写到单独的标签 `<ROUND_TAG>_test`，不会和正式结果混在一起
- 已有的测试结果会被跳过：想用新的帧范围重测，先删掉 `sleap_output/<实验>/<ROUND_TAG>_test/` 和 `reconstruction_3d/<实验>/<ROUND_TAG>_test/`
- 约 10 分钟，本格运行期间会阻塞；进度实时打印（每个视角的详细日志在 `<视角>.inference.log`）
- 看最后的 `summary` 和 `[bind] ... rows bound via top_cam_2d` 比例（round8 参考 95–97%）

In [ ]:
import subprocess

test_views = read_views(TEST_EXPERIMENT)
if test_views is None:
    raise FileNotFoundError(f"{TEST_EXPERIMENT} 缺 experiment_metadata.json，无法算出中间帧")
n_frames = min(v["total_frames"] for v in test_views.values())
fps = next(iter(test_views.values()))["frame_rate"]
half = round(TEST_SECONDS * fps / 2)
TEST_FRAMES = f"{n_frames // 2 - half}-{n_frames // 2 + half - 1}"
print(f"测试帧 {TEST_FRAMES}（共 {n_frames} 帧，{fps:g} fps，正中间 {TEST_SECONDS} 秒）\n")


def run_live(cmd):
    # 实时打印子进程输出，返回退出码
    with subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, encoding="utf-8", errors="replace", bufsize=1) as proc:
        for line in proc.stdout:
            print(line, end="")
    return proc.returncode


TEST_TAG = f"{ROUND_TAG}_test"
common = ["--paths-config", str(CFG), "--round-tag", TEST_TAG]
rc = run_live([sys.executable, "-u", "-m", "scripts.sleap.batch_bottomup_track", TEST_EXPERIMENT, *common,
               "--top-model", TOP_MODEL, "--side-model", SIDE_MODEL,
               "--batch-size", str(BATCH_SIZE), "--frames", TEST_FRAMES])
if rc != 0:
    print(f"\n❌ 推理失败，看 {PATHS.sleap_output_dir / TEST_EXPERIMENT / TEST_TAG}/<视角>.inference.log")
else:
    run_live([sys.executable, "-u", "-m", "scripts.reconstruction_3d.batch_reconstruct_3d", TEST_EXPERIMENT, *common])
    log3d = PATHS.reconstruction_output_dir / TEST_EXPERIMENT / TEST_TAG / "batch_3d.log"
    if log3d.exists():
        print("\n".join(l for l in log3d.read_text(encoding="utf-8", errors="replace").splitlines() if "[bind]" in l))

## 步骤 5 — 启动本机推理队列（后台）

- 对 `LOCAL_EXPS` 逐个实验跑 top → side1 → side2；关掉 notebook 也不会停（但重启 / 睡眠会停）
- 已完成的视角跳过；中途出错或重启电脑后，重新运行本格即可接着跑（中断的那个视角从头重跑）
- 日志：`runs_dir/batch_pipeline/local_queue.log`
- 本格也定义了步骤 9 用的 `task_alive` / `stop_background`

In [4]:
import subprocess
import time

import psutil

LOG_DIR.mkdir(parents=True, exist_ok=True)
DETACHED = subprocess.CREATE_NEW_PROCESS_GROUP | subprocess.DETACHED_PROCESS if os.name == "nt" else 0


def task_alive(name):
    # 返回仍在运行的后台进程；pid 文件里的启动时间要对得上，避免 PID 被系统复用后误判
    pid_file = LOG_DIR / f"{name}.pid.json"
    if not pid_file.exists():
        return None
    info = json.loads(pid_file.read_text())
    try:
        proc = psutil.Process(info["pid"])
        return proc if abs(proc.create_time() - info["started"]) < 60 else None
    except psutil.Error:
        return None


def start_background(name, cmd):
    if proc := task_alive(name):
        print(f"{name} 已在运行，PID {proc.pid}")
        return
    log = open(LOG_DIR / f"{name}.log", "a", encoding="utf-8")
    log.write(f"\n# started {time.ctime()}\n# cmd: {' '.join(cmd)}\n")
    log.flush()
    proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, cwd=REPO,
                            creationflags=DETACHED, start_new_session=os.name != "nt")
    (LOG_DIR / f"{name}.pid.json").write_text(json.dumps({"pid": proc.pid, "started": psutil.Process(proc.pid).create_time()}))
    print(f"{name} 已启动，PID {proc.pid}，日志 {LOG_DIR / (name + '.log')}")


def stop_background(name):
    # 连同子进程（sleap-nn / reconstruct_from_slp）一起结束
    proc = task_alive(name)
    if proc is None:
        print(f"{name} 没有在运行")
        return
    family = [proc, *proc.children(recursive=True)]
    for p in family:
        p.terminate()
    _, alive = psutil.wait_procs(family, timeout=10)
    for p in alive:
        p.kill()
    print(f"{name} 已停止（PID {proc.pid}，共 {len(family)} 个进程）")


if not RUN_LOCAL_QUEUE:
    print("未启动：确认后把步骤 1 的 RUN_LOCAL_QUEUE 改成 True，再运行本格")
else:
    start_background("local_queue", [
        sys.executable, "-u", "-m", "scripts.sleap.batch_bottomup_track", *LOCAL_EXPS,
        "--paths-config", str(CFG), "--round-tag", ROUND_TAG,
        "--top-model", TOP_MODEL, "--side-model", SIDE_MODEL, "--batch-size", str(BATCH_SIZE),
    ])

local_queue 已启动，PID 11372，日志 \\bucket.oist.jp\bucket\StephensU\medaka\runs\batch_pipeline\local_queue.log


## 步骤 6 — saion 集群提交命令

本格只**打印**命令，复制到 saion 登录节点执行（先 `git pull`，让集群上的仓库包含 bottom-up 支持——本地对 `scripts/hpc/` 的改动必须先 commit + push）。
- 每个实验 3 个视角 × `HPC_NUM_CHUNKS` 个分块并行；分块完成后自动拷回 `Z:\medaka\sleap_output\<实验>\<标签>\`
- 先用 `DRY_RUN=1` 预览一个实验，确认无误再提交全部
- **只打印还没开始的实验**：本机已能看到 `.slp` 或分块的实验直接去掉；集群上已提交过的（`/work` 里有 `merge_and_copy.log`）由循环自动跳过，所以重复粘贴也不会重复提交
- 某个实验在集群上失败、要重新提交：先在 saion 上删掉 `$MEDAKA_WORK_ROOT/<实验>/<标签>/`，再跑循环
- 分块拷回后，步骤 7 的监视器会自动合并并做 3D

In [ ]:
def inference_landed(exp):
    # 本机已能看到该实验的推理结果（合并好的 .slp 或集群拷回的分块）
    d = PATHS.sleap_output_dir / exp / ROUND_TAG
    return all((d / f"{v}.slp").exists() for v in VIEWS) or any(d.glob("*.frames_*.slp"))


HPC_TODO = [e for e in HPC_EXPS if not inference_landed(e)]
print(f"# 集群实验 {len(HPC_EXPS)} 个，其中 {len(HPC_EXPS) - len(HPC_TODO)} 个结果已拷回，待提交 {len(HPC_TODO)} 个\n")

if not HPC_TODO:
    print("# 没有需要提交的实验")
else:
    hpc_env = f'''cd {HPC_REPO} && git pull
export ROUND_TAG={ROUND_TAG}
export TOP_BOTTOMUP={HPC_MODELS_ROOT}/{TOP_MODEL}
export SIDE_BOTTOMUP={HPC_MODELS_ROOT}/{SIDE_MODEL}
export SLEAP_MODULE=sleap-nn/0.3.1 MODULEPATH_EXTRA=/apps/unit/StephensU/.modulefiles
export MAX_INSTANCES=6 BATCH_SIZE={HPC_BATCH_SIZE} NUM_CHUNKS={HPC_NUM_CHUNKS}
export PARTITION=largegpu TIME_LIMIT=12:00:00 REQUIRE_ALL_VIEWS=1
export MEDAKA_WORK_ROOT=/work/StephensU/$USER/medaka_sleap'''
    print("# ---- 1) 环境变量（每次登录先粘贴）----")
    print(hpc_env)
    print("\n# ---- 2) 预览一个 ----")
    print(f"DRY_RUN=1 bash scripts/hpc/submit_experiment.sh {HPC_TODO[0]}")
    print(f"\n# ---- 3) 提交（共 {len(HPC_TODO)} 个；已提交过的自动跳过）----")
    print("for e in " + " ".join(HPC_TODO) + "; do")
    print('    if [ -f "$MEDAKA_WORK_ROOT/$e/$ROUND_TAG/merge_and_copy.log" ]; then echo "already submitted: $e"; continue; fi')
    print('    bash scripts/hpc/submit_experiment.sh $e || echo "submit failed: $e"')
    print("done")
    print("squeue -u $USER")

## 步骤 7 — 启动两个 3D 监视器（后台）

每 10 分钟检查一次 `LOCAL_EXPS + HPC_EXPS`。三个视角的合并 `.slp` 一到，两个监视器各自开始，互不等待。

- `watch_3d`：合并集群分块 → 2D 追踪 → 旧 3D（`reconstruct_from_slp`），约 10–30 分钟
- `watch_m3d`：新 3D（m3dfull：c27 → c34 → `<标签>_m3dfull` 和审阅包），一次一个实验，约 2 小时。已有 CSV 和审阅包的跳过
- 失败的会记在日志里，**本次运行内不会重试**：修好后用步骤 9 停掉对应监视器，再运行本格
- 日志：`runs_dir/batch_pipeline/watch_3d.log`、`watch_m3d.log`；旧 3D 详情在 `reconstruction_3d/<实验>/<标签>/batch_3d.log`，新 3D 详情在 `work/medaka3d_chain_logs/<实验>/`
- 集群分块还没合并成 `<视角>.slp` 时，新 3D 会一直显示等待；合并由 `watch_3d` 做

In [5]:
if not RUN_3D_WATCHER:
    print("未启动：确认后把步骤 1 的 RUN_3D_WATCHER 改成 True，再运行本格")
else:
    start_background("watch_3d", [
        sys.executable, "-u", "-m", "scripts.reconstruction_3d.batch_reconstruct_3d", *LOCAL_EXPS, *HPC_EXPS,
        "--paths-config", str(CFG), "--round-tag", ROUND_TAG, "--watch", "10",
    ])
    start_background("watch_m3d", [
        sys.executable, "-u", "-m", "scripts.reconstruction_3d.medaka3d_batch", *LOCAL_EXPS, *HPC_EXPS,
        "--round-tag", ROUND_TAG, "--jobs", "1", "--watch", "10",
    ])

watch_3d 已启动，PID 11296，日志 \\bucket.oist.jp\bucket\StephensU\medaka\runs\batch_pipeline\watch_3d.log


## 步骤 8 — 进度总表

随时重新运行（先运行步骤 1–3、5，本格要用 `task_alive`）。
- 每个视角：✅ 已合并 `.slp`；🧩 集群分块已到；⏳ 本机正在跑；❌ 本机跑过但没有结果（失败或被中断，看 `<视角>.inference.log`）；空白 = 未开始 / 集群还没拷回
- 3D：旧的 `reconstruct_from_slp`。✅ 完成（有 `batch_3d.done`）；❌ 最近一次失败（看 `batch_3d.log`）；▶ 正在跑
- 新3D：m3dfull。✅ 已有 `<_3d.csv` 和审阅包；▶ 监视器正在跑；❌ 跑过但没有结果（看 `work/medaka3d_chain_logs/<实验>/batch.log`）；空白 = 还在等推理

In [16]:
import pandas as pd

from scripts.reconstruction_3d.batch_reconstruct_3d import DONE_MARKER
from scripts.sleap.batch_bottomup_track import lock_is_live


def view_status(slp_dir, v):
    if (slp_dir / f"{v}.slp").exists():
        return "✅"
    if any(slp_dir.glob(f"{v}.frames_*.slp")):
        return "🧩"
    if lock_is_live(slp_dir / f"{v}.running"):
        return "⏳"
    return "❌" if (slp_dir / f"{v}.inference.log").exists() else ""


def recon_status(exp):
    out_dir = PATHS.reconstruction_output_dir / exp / ROUND_TAG
    if (out_dir / DONE_MARKER).exists():
        return "✅"
    log = out_dir / "batch_3d.log"
    if not log.exists():
        return ""
    last = [l for l in log.read_text(encoding="utf-8", errors="replace").splitlines() if l.startswith(("# cmd:", "# exit"))]
    if not last:
        return ""
    if last[-1].startswith("# cmd:"):
        return "▶" if task_alive("watch_3d") else "❌"   # 开始了但没有结束记录：监视器没在跑 = 被中断
    return "❌" if last[-1].split()[2] != "0" else ""


def m3d_status(exp):
    tag = f"{ROUND_TAG}_m3dfull"
    csv = PATHS.reconstruction_output_dir / exp / tag / f"{tag}_3d.csv"
    review = PATHS.data_root / "review_packages" / exp / tag / "review_manifest.json"
    if csv.is_file() and review.is_file():
        return "✅"
    log = PATHS.data_root / "work" / "medaka3d_chain_logs" / exp / "batch.log"
    if not log.exists():
        return ""
    text = log.read_text(encoding="utf-8", errors="replace")
    tail = text[text.rfind("# started"):]
    if "done ->" in tail:
        return "❌"
    return "▶" if task_alive("watch_m3d") else "❌"


rows = []
for exp in LOCAL_EXPS + HPC_EXPS:
    slp_dir = PATHS.sleap_output_dir / exp / ROUND_TAG
    row = {"实验": exp, "分工": "本机" if exp in LOCAL_EXPS else "集群"}
    row.update({v: view_status(slp_dir, v) for v in VIEWS})
    row["3D"] = recon_status(exp)
    row["新3D"] = m3d_status(exp)
    rows.append(row)
df = pd.DataFrame(rows)
print(f"推理完成 {(df[list(VIEWS)] == '✅').all(axis=1).sum()}/{len(df)}，"
      f"旧 3D 完成 {(df['3D'] == '✅').sum()}/{len(df)}，新 3D 完成 {(df['新3D'] == '✅').sum()}/{len(df)}")
n_fail = (df[[*VIEWS, "3D", "新3D"]] == "❌").any(axis=1).sum()
if n_fail:
    print(f"⚠️ {n_fail} 个实验有失败项（❌）")
for name in ("local_queue", "watch_3d", "watch_m3d"):
    print(f"\n{name}：{'运行中' if task_alive(name) else '未运行'}")
    log = LOG_DIR / f"{name}.log"
    if log.exists():
        print("\n".join(log.read_text(encoding="utf-8", errors="replace").splitlines()[-4:]))
df

推理完成 4/6，3D 完成 4/6

local_queue：运行中
[10-28 05:46] (4/6) 20260118_P_2_1_FCMC side1_cam: done (3.00 h)
[10-28 05:46] (4/6) 20260118_P_2_1_FCMC side2_cam ...
[10-28 08:47] (4/6) 20260118_P_2_1_FCMC side2_cam: done (3.02 h)
[10-28 08:47] (5/6) 20260118_P_2_2_FCMC top_cam ...

watch_3d：运行中
[10-28 10:13] 2 waiting; next check in 10 min
[10-28 10:23] 20260118_P_2_2_FCMC: waiting for inference
[10-28 10:23] 20260118_P_2_3_FCMC: waiting for inference
[10-28 10:23] 2 waiting; next check in 10 min


,实验,分工,top_cam,side1_cam,side2_cam,3D
0,20260112_P_1_1_FCMC,本机,✅,✅,✅,✅
1,20260112_P_1_3_FCMC,本机,✅,✅,✅,✅
2,20260113_P_1_2_FCMC,本机,✅,✅,✅,✅
3,20260118_P_2_1_FCMC,本机,✅,✅,✅,✅
4,20260118_P_2_2_FCMC,本机,⏳,,,
5,20260118_P_2_3_FCMC,本机,,,,


## 步骤 9 — 停止后台任务

需要重启 3D 监视器（例如修好失败的实验后）或暂停本机推理时用。先运行步骤 1–3、5。
- 把 `STOP_TASK` 改成 `"watch_3d"`、`"watch_m3d"` 或 `"local_queue"` 再运行；会连同子进程一起结束
- 停掉 `local_queue` 时，正在跑的那个视角作废，下次启动从头重跑

In [ ]:
STOP_TASK = None   # "watch_3d"、"watch_m3d" 或 "local_queue"

if STOP_TASK is None:
    print("未停止：把 STOP_TASK 改成要停的任务名再运行本格")
else:
    stop_background(STOP_TASK)

## 步骤 10 — 用 Rerun 查看 3D 结果

打开 Rerun 窗口：三个相机视频同步播放，叠加 2D 检测点和 3D 轨迹，可以拖时间轴、旋转 3D。先运行步骤 1–2。
- `VIEW_TAG`：小测试用 `f"{ROUND_TAG}_test"`，正式结果用 `ROUND_TAG`
- 只看一小段（默认 30 秒）：全长视频太大，加载会非常慢
- 小测试已有结果的帧范围是 `171800` 起 30 秒；正式结果任意帧都可以
- `VIEW_NO_VIDEO = True` 只看 3D 和诊断曲线，不加载视频，快很多
- 鱼的颜色按 `fish_id`（ID_1/2/3）区分，和 CSV 里的 `final_fish_id` 编号不是同一套

In [ ]:
import subprocess

VIEW_EXPERIMENT  = TEST_EXPERIMENT
VIEW_TAG         = f"{ROUND_TAG}_test"
VIEW_START_FRAME = 171800
VIEW_SECONDS     = 30
VIEW_NO_VIDEO    = False

rec_dir = PATHS.reconstruction_output_dir / VIEW_EXPERIMENT / VIEW_TAG
if not (rec_dir / "batch_3d.done").exists():
    raise FileNotFoundError(f"还没有完成的 3D 结果：{rec_dir}")
cmd = [sys.executable, "-u", "-m", "scripts.visualization.rerun_viewer", VIEW_EXPERIMENT,
       "--paths-config", str(CFG),
       "--csv", str(rec_dir / f"{VIEW_TAG}_3d_identified.csv"),
       "--best-json", str(rec_dir / f"{VIEW_TAG}_best.json"),
       "--pre-link-json", str(rec_dir / f"{VIEW_TAG}_best_pre_link.json"),
       "--slp-dir", str(PATHS.sleap_output_dir / VIEW_EXPERIMENT / VIEW_TAG),
       "--start-frame", str(VIEW_START_FRAME),
       "--end-frame", str(VIEW_START_FRAME + round(VIEW_SECONDS * 166) - 1)]
if VIEW_NO_VIDEO:
    cmd.append("--no-video")
# 加载完数据后 Rerun 窗口会自己弹出；本格结束后窗口仍然保留
with subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                      text=True, encoding="utf-8", errors="replace", bufsize=1) as proc:
    for line in proc.stdout:
        print(line, end="")